In [1]:
import gradio as gr

In [2]:
def greet(name):
    return "Hello " + name + "!!"

In [3]:
demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv


So what happened here, Gradio took the function as input and then we told it what are the inputs and the outputs with it.

Lets try one more example

In [ ]:
def shout(name):
    return name.upper()

demo_shout = gr.Interface(fn=shout,inputs="text",outputs="text",flagging_mode="never") #The flag button is not visible
demo_shout.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [5]:
#Lets play around this with LLMs now

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [8]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

if not openai_api_key:
    print("No OpenAI API key found")
else:
    print("OpenAI API key found")

if not anthropic_api_key:
    print("No Anthropic API key found")
else:
    print("Anthropic API key found")


OpenAI API key found
Anthropic API key found


In [9]:
anthropic_url = "https://api.anthropic.com/v1/"

openai_client = OpenAI()
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

Use Case 1: 
- If we always want to open the result in a browser, we would use .launch(ibrowser=true)
- If we want to end up sharing the result on a server for a week for free, we will use .launch(share=true)
- If we do not want to have the flagging, we use gr.Interface(flagging_mode="never")
- If we want to have authentication set in, we will use .alunch

In [16]:
def chat_with_llm(model_name, prompt):
    result = ""
    SYSTEM_PROMPT = """ 
        You are a goofy assistant who likes to help users with their queries.
    """
    if model_name == "OpenAI":
        response = openai_client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                "role": "system", "content": SYSTEM_PROMPT,
                "role": "user", "content": prompt
                }
            ]
        )
        result = response.choices[0].message.content
    else:
        response = anthropic_client.chat.completions.create(
            model="claude-sonnet-4-5-20250929",
            messages=[
                {
                "role": "system", "content": SYSTEM_PROMPT,
                "role": "user", "content": prompt
                }
            ]
        )
        result = response.choices[0].message.content
    return result

In [17]:
#What I want to do is I want to have a gradio interface where I can have a dropdown for the model name and a text box for the prompt.
#Also I will add authenitcation to this and host this to an external website via the share=True option

message_to_llm = gr.Textbox(label="Whatg do you want to ask the LLM?",lines=5)
llm_selector = gr.Dropdown(choices=["OpenAI", "Anthropic"], value="OpenAI", label="Select the LLM")
llm_response = gr.Markdown(label="Response from the LLM")

demo = gr.Interface(
    fn=chat_with_llm,
    inputs=[llm_selector, message_to_llm],
    outputs=llm_response,
    allow_flagging="never",
    examples=[
        ["OpenAI", "Explain the concept of LLMs in layman's terms?"],
        ["Anthropic", "Explain the concept of LLMs in layman's terms?"]
    ]
).launch(auth=("jigar", "jigar123"),inbrowser=True,share=True)


c:\Users\Jigar Joshi\Desktop\AI 2026\LLM Engineer Practice\LLM-Engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://6e6ff8f60eaa9e5962.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
#Lets take this one step further and add streaming to this so that the user can see the response as it is being generated

def chat_with_llm_streaming(model_name, prompt):
    result = ""
    SYSTEM_PROMPT = """ 
        You are a goofy assistant who likes to help users with their queries.
    """
    if model_name == "OpenAI":
        response = openai_client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                "role": "system", "content": SYSTEM_PROMPT,
                "role": "user", "content": prompt
                }
            ],
            stream=True
        )
    else:
        response = anthropic_client.chat.completions.create(
            model="claude-sonnet-4-5-20250929",
            messages=[
                {
                "role": "system", "content": SYSTEM_PROMPT,
                "role": "user", "content": prompt
                }
            ],
            stream=True
        )
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [19]:
message_to_llm = gr.Textbox(label="Whatg do you want to ask the LLM?",lines=5)
llm_selector = gr.Dropdown(choices=["OpenAI", "Anthropic"], value="OpenAI", label="Select the LLM")
llm_response = gr.Markdown(label="Response from the LLM")

gr.Interface(
    title="Chat with LLM",
    description="Ask the LLM anything you want",
    fn=chat_with_llm_streaming,
    inputs=[llm_selector, message_to_llm],
    outputs=llm_response,
    allow_flagging="never",
    examples=[
        ["OpenAI", "Explain the concept of LLMs in layman's terms?"],
        ["Anthropic", "Explain the concept of LLMs in layman's terms?"]
    ]
).launch(inbrowser=True,share=True)


c:\Users\Jigar Joshi\Desktop\AI 2026\LLM Engineer Practice\LLM-Engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://c0136b40e48f6d7ebe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
